# Fine-Tuning d'un modèle de langage sur un corpus technique

**Projet académique - M1 Mathématiques Appliquées au Calcul et à l'IA (MACIA)**

---

## Objectif

Ce projet illustre le processus complet de fine-tuning d'un modèle de langage pré-entraîné (DistilGPT-2) sur un corpus technique spécialisé en aérospatiale. L'objectif est de démontrer la maîtrise des concepts suivants :

- Extraction et prétraitement de données textuelles
- Tokenization et gestion des séquences
- Fine-tuning de transformers avec Hugging Face
- Évaluation qualitative de modèles génératifs

---

## Architecture utilisée

- **Modèle** : DistilGPT-2 (82M paramètres)
- **Framework** : Hugging Face Transformers
- **Tâche** : Causal Language Modeling (CLM)
- **Domaine** : Textes techniques en aérospatiale

## 1. Installation des dépendances

In [1]:
# Installation des bibliothèques nécessaires
!pip install transformers datasets accelerate -Uq
!pip install torch wikipedia -q
!pip install torch matplotlib -q


# Vérification du GPU disponible
import torch
print(f"\nGPU disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Exécution sur CPU")


GPU disponible: False
Exécution sur CPU


## 2. Construction du corpus de données

Pour ce projet, nous construisons un corpus spécialisé à partir de Wikipedia en extrayant des articles techniques sur l'aérospatiale. Cette approche permet d'obtenir un dataset ciblé et à jour.

In [2]:
import wikipedia
import re
import os

# Fix: Configuration d'un User-Agent pour éviter l'erreur 403 de Wikipedia
wikipedia.set_user_agent("Portfolio-IA-Bot/1.0 (Portfolio Project; frankdanielou@gmail.com)")

# Fix: Éviter le crash du kernel à cause de libiomp5md.dll
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# Configuration de l'API Wikipedia en anglais
wikipedia.set_lang("en")

# Définition des sujets techniques
topics = [
    "Ariane 6",
    "Vulcain (rocket engine)",
    "Vinci (rocket engine)",
    "Rocket propulsion",
    "Orbital mechanics",
    "Hohmann transfer orbit",
    "Specific impulse",
    "Cryogenic rocket engine",
    "Space debris"
]

print("Extraction des données depuis Wikipedia...\n")

# Extraction du contenu des articles
raw_text = ""
for topic in topics:
    try:
        page = wikipedia.page(topic, auto_suggest=False)
        content = page.content
        print(f"  ✓ {topic}: {len(content)} caractères")
        raw_text += content + "\n\n"
    except Exception as e:
        print(f"  ✗ Erreur sur {topic}: {e}")

# Nettoyage du texte
# Suppression des références Wikipedia (ex: [12], [edit])
clean_text = re.sub(r'\[.*?\]', '', raw_text)
# Normalisation des sauts de ligne
clean_text = re.sub(r'\n\s*\n', '\n\n', clean_text)

# Sauvegarde du corpus
file_path = "aerospace_corpus.txt"
with open(file_path, "w", encoding="utf-8") as f:
    f.write(clean_text)

print(f"\n✓ Corpus sauvegardé: {file_path}")
print(f"  Taille totale: {len(clean_text):,} caractères")
print(f"  Nombre de mots (approx.): {len(clean_text.split()):,}")
print(f"\n{'='*60}")
print("Aperçu du corpus:")
print(f"{'='*60}\n{clean_text[:500]}\n{'='*60}")

Extraction des données depuis Wikipedia...



  ✓ Ariane 6: 24716 caractères


  ✓ Vulcain (rocket engine): 4354 caractères


  ✓ Vinci (rocket engine): 3978 caractères


  ✓ Rocket propulsion: 37943 caractères


  ✓ Orbital mechanics: 64670 caractères


  ✓ Hohmann transfer orbit: 26397 caractères


  ✓ Specific impulse: 33630 caractères


  ✓ Cryogenic rocket engine: 3166 caractères


  ✓ Space debris: 65575 caractères

✓ Corpus sauvegardé: aerospace_corpus.txt
  Taille totale: 236,291 caractères
  Nombre de mots (approx.): 34,791

Aperçu du corpus:
Ariane 6 (French: ) is a European expendable launch system developed for the European Space Agency (ESA) and French Space Agency (CNES) and manufactured by a consortium of European companies, led by the prime contractor ArianeGroup. As part of the Ariane rocket family, it is operated by Arianespace, replacing the Ariane 5. The project's primary contributors were France (55.3%), Germany (21%) and Italy (7.6%), with the remaining work distributed among ten other participating countries.
This two-s


## 3. Tokenization et préparation des données

La tokenization est l'étape de conversion du texte en séquences d'entiers compréhensibles par le modèle. DistilGPT-2 utilise un tokenizer BPE (Byte Pair Encoding) avec un vocabulaire de 50,257 tokens.

**Focus théorique : Le Byte Pair Encoding (BPE)**

Le choix du tokenizer n'est pas anodin. Le BPE est un algorithme de compression de données qui trouve un compromis élégant entre une tokenization par mot (vocabulaire immense, gestion difficile des mots inconnus) et par caractère (séquences trop longues, perte de sens). Il fusionne itérativement les paires de caractères les plus fréquentes pour construire un vocabulaire sous-mot. Cela permet au modèle de traiter efficacement le vocabulaire technique aérospatial même s'il n'a pas rencontré tous les mots exacts lors de son pré-entraînement.

In [3]:
from transformers import AutoTokenizer

# Chargement du tokenizer DistilGPT-2
model_checkpoint = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# Configuration du token de padding
# GPT-2 n'a pas de pad_token par défaut, on utilise le token EOS
tokenizer.pad_token = tokenizer.eos_token

# Test de tokenization
sample_text = "The Vulcain engine uses liquid hydrogen."
tokens = tokenizer(sample_text)

print(f"Texte original: '{sample_text}'")
print(f"\nTokens ID: {tokens['input_ids']}")
print(f"\nDécodage token par token:")
for i, token_id in enumerate(tokens['input_ids']):
    print(f"  {i+1}. [{token_id:5d}] → '{tokenizer.decode([token_id])}'")

print(f"\nVocabulaire: {tokenizer.vocab_size:,} tokens")

Texte original: 'The Vulcain engine uses liquid hydrogen.'

Tokens ID: [464, 25442, 66, 391, 3113, 3544, 8122, 17669, 13]

Décodage token par token:
  1. [  464] → 'The'
  2. [25442] → ' Vul'
  3. [   66] → 'c'
  4. [  391] → 'ain'
  5. [ 3113] → ' engine'
  6. [ 3544] → ' uses'
  7. [ 8122] → ' liquid'
  8. [17669] → ' hydrogen'
  9. [   13] → '.'

Vocabulaire: 50,257 tokens


## 4. Configuration et fine-tuning du modèle

Le fine-tuning consiste à poursuivre l'entraînement d'un modèle pré-entraîné sur un corpus spécifique. Nous utilisons l'approche Causal Language Modeling (CLM) : le modèle apprend à prédire le token suivant étant donné un contexte.

### Fonction de perte

Pour chaque séquence, le modèle calcule la distribution de probabilité sur le vocabulaire et optimise la cross-entropie :

$$\mathcal{L} = -\sum_{t=1}^{T} \log P(w_t | w_{<t})$$

où $w_t$ est le token à la position $t$ et $w_{<t}$ le contexte précédent.

### Optimisation

La mise à jour des poids se fait par descente de gradient stochastique (SGD) avec Adam :

$$\theta_{t+1} = \theta_t - \eta \cdot \nabla_\theta \mathcal{L}$$

où $\eta$ est le learning rate et $\theta$ les paramètres du modèle.

**Pourquoi des blocs de 128 tokens ?**

Dans notre approche, les textes sont concaténés puis découpés en fenêtres de taille fixe (128 tokens). Cette méthode, classique en Causal Language Modeling (CLM), garantit que le modèle reçoit des tenseurs de dimensions constantes sans avoir recours à un remplissage excessif (padding). Bien que l'on perde occasionnellement le contexte exact aux frontières des blocs, cette perte est négligeable face au gain d'efficacité computationnelle lors de l'entraînement.

In [4]:
import torch
from transformers import (
    DataCollatorForLanguageModeling,
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments
)
from datasets import load_dataset

# 1. Chargement du tokenizer
model_checkpoint = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
tokenizer.pad_token = tokenizer.eos_token

# 2. Chargement du modèle pré-entraîné
print(f"Chargement du modèle {model_checkpoint}...")
model = AutoModelForCausalLM.from_pretrained(model_checkpoint)

# Transfert sur GPU si disponible
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print(f"  Modèle chargé sur {device}")
print(f"  Nombre de paramètres: {model.num_parameters():,}")

# 3. Préparation du dataset
# Utilisation de la bibliothèque datasets pour charger le corpus
print("\nChargement du corpus avec datasets...")
raw_datasets = load_dataset('text', data_files={'train': 'aerospace_corpus.txt'})

# Le corpus est découpé en blocs de 128 tokens pour l'entraînement
block_size = 128

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True)

# Tokenization du corpus
tokenized_datasets = raw_datasets.map(
    tokenize_function,
    batched=True,
    num_proc=4, # Utiliser plusieurs processus pour accélérer la tokenization
    remove_columns=["text"]
)

def group_texts(examples):
    # Concaténer tous les textes. Il y a un problème avec l'affichage de ce caractère unicode.
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    # Tronquer pour s'assurer que la longueur totale est un multiple de block_size
    total_length = (total_length // block_size) * block_size
    # Diviser en blocs de block_size
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

# Regroupement des tokens en blocs de taille fixe
train_dataset = tokenized_datasets.map(
    group_texts,
    batched=True,
    num_proc=4
) # Utiliser num_proc pour accélérer le traitement

print(f"\n  Nombre d'exemples d'entraînement: {len(train_dataset['train'])}")

# 4. Data collator pour le masquage causal
# mlm=False indique que nous faisons du Causal LM (pas du Masked LM comme BERT)
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# 5. Configuration des hyperparamètres d'entraînement
training_args = TrainingArguments(
    output_dir="./aerospace_model",
    num_train_epochs=3,              # Nombre d'époques
    per_device_train_batch_size=8,   # Taille de batch
    learning_rate=5e-5,              # Taux d'apprentissage
    save_steps=500,                  # Sauvegarde tous les 500 steps
    save_total_limit=2,              # Garde seulement 2 checkpoints
    logging_steps=50,                # Log tous les 50 steps
    report_to="none"                 # Désactive les rapports externes
)

# 6. Initialisation du Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset['train'] # Passer la split 'train' du dataset
)

# 7. Lancement du fine-tuning
print("\nDémarrage du fine-tuning...\n")
trainer.train()

# 8. Sauvegarde du modèle fine-tuné
output_dir = "./aerospace_model_final"
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"\n✓ Modèle sauvegardé dans {output_dir}")

Chargement du modèle distilgpt2...


  Modèle chargé sur cpu
  Nombre de paramètres: 81,912,576

Chargement du corpus avec datasets...


Generating train split: 0 examples [00:00, ? examples/s]

Map (num_proc=4):   0%|          | 0/3694 [00:00<?, ? examples/s]

NameError: name 'tokenizer' is not defined

### Visualisation de la convergence

Pour s'assurer que le modèle a bien appris la distribution de notre corpus, on trace l'évolution de la fonction de perte (Cross-Entropy Loss) sur nos étapes d'entraînement. Une courbe décroissante valide notre approche d'optimisation.

In [5]:
import matplotlib.pyplot as plt

# Extraction de l'historique d'entraînement du Trainer
history = trainer.state.log_history

# On filtre pour ne garder que les logs contenant la 'loss' (les logs d'entraînement)
steps = [log['step'] for log in history if 'loss' in log]
losses = [log['loss'] for log in history if 'loss' in log]

plt.figure(figsize=(10, 5))
plt.plot(steps, losses, marker='o', linestyle='-', color='b', linewidth=2, markersize=6)
plt.title('Évolution de la Training Loss', fontsize=14)
plt.xlabel('Étapes (Steps)', fontsize=12)
plt.ylabel('Cross-Entropy Loss', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

NameError: name 'trainer' is not defined

## 5. Évaluation et génération de texte

Une fois le modèle fine-tuné, nous évaluons sa capacité à générer du texte cohérent dans le domaine technique visé.

### Paramètres de génération

- **temperature** : Contrôle l'aléatoire de la génération ($T \in [0,1]$). Une température basse rend le modèle plus déterministe.
- **repetition_penalty** : Pénalise les tokens déjà générés pour éviter les répétitions.
- **no_repeat_ngram_size** : Interdit la répétition de n-grammes de taille donnée.

In [6]:
from transformers import pipeline

# Chargement du modèle fine-tuné
model_path = "./aerospace_model_final"
generator = pipeline(
    'text-generation',
    model=model_path,
    tokenizer=model_path,
    device=0 if torch.cuda.is_available() else -1
)

# Prompts de test
test_prompts = [
    "The Ariane 6 launcher is designed to",
    "The Vulcain engine uses liquid oxygen and",
    "Orbital mechanics describes the motion of"
]

print("="*70)
print("GÉNÉRATION DE TEXTE - MODÈLE FINE-TUNÉ")
print("="*70)

for i, prompt in enumerate(test_prompts, 1):
    print(f"\n[Test {i}]")
    print(f"Prompt: '{prompt}'")
    print("-"*70)

    result = generator(
        prompt,
        max_new_tokens=50,
        num_return_sequences=1,
        temperature=0.7,
        repetition_penalty=1.5,
        no_repeat_ngram_size=3,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    generated_text = result[0]['generated_text']
    print(f"Texte généré:\n{generated_text}")
    print("="*70)

HFValidationError: Repo id must use alphanumeric chars, '-', '_' or '.'. The name cannot start or end with '-' or '.' and the maximum length is 96: './aerospace_model_final'.

### Évaluation comparative : Avant vs Après

La véritable preuve de l'efficacité du fine-tuning réside dans la comparaison directe. Nous allons charger le modèle `distilgpt2` d'origine (non entraîné sur notre corpus) et lui soumettre les mêmes prompts. Cela nous permettra de constater empiriquement le changement de registre sémantique.

In [7]:
from transformers import pipeline

# Chargement du modèle de base pour comparaison
base_generator = pipeline(
    'text-generation',
    model='distilgpt2',
    tokenizer='distilgpt2',
    device=0 if torch.cuda.is_available() else -1
)

print("="*70)
print("COMPARAISON DIRECTE : BASE vs FINE-TUNED")
print("="*70)

for i, prompt in enumerate(test_prompts, 1):
    print(f"\n[Test {i} - Prompt: '{prompt}']")
    print("-" * 70)
    
    # Génération Base
    res_base = base_generator(prompt, max_new_tokens=40, num_return_sequences=1, temperature=0.7, pad_token_id=tokenizer.eos_token_id)
    print(f"DistilGPT-2 (Base) :\n{res_base[0]['generated_text']}")
    print("-" * 40)
    
    # Génération Fine-Tuned
    res_ft = generator(prompt, max_new_tokens=40, num_return_sequences=1, temperature=0.7, pad_token_id=tokenizer.eos_token_id)
    print(f"DistilGPT-2 (Spécialisé) :\n{res_ft[0]['generated_text']}")


Device set to use cpu


COMPARAISON DIRECTE : BASE vs FINE-TUNED


NameError: name 'test_prompts' is not defined

## 6. Conclusion et Perspectives

Ce projet illustre de bout en bout la démarche d'adaptation de domaine pour un grand modèle de langage. Au-delà de la simple exécution technique, il démontre que des modèles relativement légers (comme DistilGPT-2) peuvent capter une sémantique très spécialisée si on leur fournit un corpus de qualité, sans nécessiter d'infrastructures de calcul démesurées.

### Limites de l'approche et pistes d'amélioration

L'approche adoptée ici, bien que fonctionnelle, correspond à un *full fine-tuning*, où l'intégralité des poids du modèle est mise à jour. C'est une stratégie assez coûteuse computationnellement.

Pour une évolution naturelle de ce projet, je me tournerais aujourd'hui vers les méthodes **PEFT (Parameter-Efficient Fine-Tuning)**, et plus particulièrement **LoRA (Low-Rank Adaptation)**. Au lieu de recalculer tous les paramètres, on gèle le modèle pré-entraîné et on injecte des matrices de rang faible dans les couches d'attention. Couplé à de la quantification sur 4 bits (**QLoRA**), cela permettrait de spécialiser des modèles beaucoup plus massifs (comme Llama-3 8B) sur des machines grand public, ouvrant la porte à des performances de pointe.